# KSP Enriched Company Analysis

Analysis of 867 companies enriched with internal order features and Companies House data.

**Data source:** `data/companies/company_features.csv` (98 features per company)

**Enrichment sources:**
- Internal: RFM metrics, product mix, operations, pricing, seasonality from 8,078 legacy estimates
- Companies House basic: industry sector (SIC codes), company age, region, type, status, accounts category
- Companies House deep: officers, directors, filing history, charges (secured lending), PSC
- Web presence: domain detection, HTTPS status

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 85)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

df = pd.read_csv(Path("../data/companies/company_features.csv"))
print(f"Dataset: {df.shape[0]} companies × {df.shape[1]} features")
df.head(3)

## 1. Enrichment Coverage & Data Quality

In [ ]:
# Overall field completeness
completeness = df.notna().mean().sort_values(ascending=False)

print("Field completeness:")
for col, pct in completeness.items():
    bar = "█" * int(pct * 30)
    print(f"  {col:45s} {pct:5.1%}  {bar}")

In [ ]:
# Companies House match quality
ch_cols = ["ch_company_name", "company_number", "company_type", "company_status",
           "sic_codes", "industry_sector", "incorporation_date", "company_age_years",
           "region", "accounts_type"]

print("Companies House enrichment coverage:")
for c in ch_cols:
    n = df[c].notna().sum()
    print(f"  {c:30s} {n:>4}/{len(df)}  ({100*n/len(df):.1f}%)")

# Unmatched companies
unmatched = df[df["company_number"].isna()]
print(f"\nUnmatched companies ({len(unmatched)}):")
for _, row in unmatched.iterrows():
    print(f"  {row['company']:40s}  orders={row['frequency']}  revenue=£{row['monetary_total']:,.0f}")

## 2. Industry Sector Analysis

In [ ]:
# Revenue and volume by industry sector
sector = df[df["industry_sector"].notna()].groupby("industry_sector").agg(
    companies=("company", "count"),
    total_revenue=("monetary_total", "sum"),
    avg_revenue=("monetary_total", "mean"),
    median_revenue=("monetary_total", "median"),
    avg_frequency=("frequency", "mean"),
    avg_quantity=("avg_quantity", "mean"),
    avg_margin=("avg_margin", "mean"),
    avg_operations=("avg_num_operations", "mean"),
).sort_values("total_revenue", ascending=False).reset_index()

sector["revenue_pct"] = (sector["total_revenue"] / sector["total_revenue"].sum() * 100).round(1)
sector["company_pct"] = (sector["companies"] / sector["companies"].sum() * 100).round(1)

print("Revenue by industry sector:")
sector

In [ ]:
# Revenue concentration — which sectors punch above their weight?
sector["revenue_vs_company_ratio"] = (sector["revenue_pct"] / sector["company_pct"]).round(2)
print("Revenue/company ratio (>1 = higher revenue per company):")
print(sector[["industry_sector", "companies", "company_pct", "revenue_pct", "revenue_vs_company_ratio"]]
      .sort_values("revenue_vs_company_ratio", ascending=False)
      .to_string(index=False))

In [ ]:
# Product preferences by top sectors
ptype_cols = [c for c in df.columns if c.startswith("ptype_")]
top_sectors = sector.head(6)["industry_sector"].tolist()

sector_products = df[df["industry_sector"].isin(top_sectors)].groupby("industry_sector")[ptype_cols].mean().round(1)
sector_products.columns = [c.replace("ptype_", "").replace("_pct", "") for c in sector_products.columns]
print("Product type preferences by top sector (%):")
sector_products

## 3. Geographic Analysis

In [ ]:
# Revenue and volume by region
regional = df[df["region"].notna()].groupby("region").agg(
    companies=("company", "count"),
    total_revenue=("monetary_total", "sum"),
    avg_revenue=("monetary_total", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_margin=("avg_margin", "mean"),
).sort_values("total_revenue", ascending=False).reset_index()

regional["revenue_pct"] = (regional["total_revenue"] / regional["total_revenue"].sum() * 100).round(1)

print(f"Regions represented: {len(regional)}")
print("\nTop 20 regions by revenue:")
regional.head(20)

In [ ]:
# London vs rest
london = df[df["region"] == "London"]
non_london = df[(df["region"].notna()) & (df["region"] != "London")]

print("London vs Rest of UK:")
print(f"  {'':30s} {'London':>15s} {'Rest':>15s}")
print(f"  {'Companies':30s} {len(london):>15,} {len(non_london):>15,}")
print(f"  {'Total revenue':30s} £{london['monetary_total'].sum():>14,.0f} £{non_london['monetary_total'].sum():>14,.0f}")
print(f"  {'Avg revenue per company':30s} £{london['monetary_total'].mean():>14,.0f} £{non_london['monetary_total'].mean():>14,.0f}")
print(f"  {'Avg frequency':30s} {london['frequency'].mean():>15.1f} {non_london['frequency'].mean():>15.1f}")
print(f"  {'Avg margin':30s} {london['avg_margin'].mean():>14.1f}% {non_london['avg_margin'].mean():>14.1f}%")

## 4. Company Maturity & Size Analysis

In [ ]:
# Company age distribution and its relationship to order behaviour
aged = df[df["company_age_years"].notna()].copy()

age_bins = [0, 2, 5, 10, 20, 50, 200]
age_labels = ["0-2yr", "2-5yr", "5-10yr", "10-20yr", "20-50yr", "50yr+"]
aged["age_band"] = pd.cut(aged["company_age_years"], bins=age_bins, labels=age_labels)

age_stats = aged.groupby("age_band", observed=True).agg(
    companies=("company", "count"),
    total_revenue=("monetary_total", "sum"),
    avg_revenue=("monetary_total", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_quantity=("avg_quantity", "mean"),
    avg_margin=("avg_margin", "mean"),
    avg_operations=("avg_num_operations", "mean"),
).reset_index()

print("Order behaviour by company age:")
age_stats

In [ ]:
# Company size proxy via accounts type
# accounts_type hierarchy: micro < total-exemption-small < small < total-exemption-full < medium < full < group
size_order = {
    "dormant": "Dormant",
    "micro-entity": "Micro",
    "total-exemption-small": "Small (exempt)",
    "small": "Small",
    "total-exemption-full": "Full (exempt)",
    "audit-exemption-subsidiary": "Subsidiary",
    "medium": "Medium",
    "unaudited-abridged": "Abridged",
    "full": "Full",
    "group": "Group",
}

with_acct = df[df["accounts_type"].notna()].copy()
with_acct["size_label"] = with_acct["accounts_type"].map(size_order).fillna(with_acct["accounts_type"])

size_stats = with_acct.groupby("size_label").agg(
    companies=("company", "count"),
    total_revenue=("monetary_total", "sum"),
    avg_revenue=("monetary_total", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_margin=("avg_margin", "mean"),
).sort_values("total_revenue", ascending=False).reset_index()

print("Order behaviour by company size (accounts type proxy):")
size_stats

In [ ]:
# Company status breakdown — active vs dissolved vs other
status_stats = df[df["company_status"].notna()].groupby("company_status").agg(
    companies=("company", "count"),
    total_revenue=("monetary_total", "sum"),
    avg_revenue=("monetary_total", "mean"),
    avg_recency=("recency_days", "mean"),
).sort_values("total_revenue", ascending=False).reset_index()

print("Order behaviour by company status:")
status_stats

## 5. Customer Value Segments

In [ ]:
# Cross-tab: value quintile vs frequency tier
xtab = pd.crosstab(df["value_quintile"], df["frequency_tier"], margins=True)
print("Value quintile × Frequency tier:")
xtab

In [ ]:
# Churn risk: active vs churned by value quintile
churn_by_value = df.groupby("value_quintile").agg(
    total=("company", "count"),
    active_12m=("is_active_12m", "sum"),
    active_6m=("is_active_6m", "sum"),
    churned=("is_churned", "sum"),
).reset_index()
churn_by_value["active_12m_pct"] = (churn_by_value["active_12m"] / churn_by_value["total"] * 100).round(1)
churn_by_value["churn_pct"] = (churn_by_value["churned"] / churn_by_value["total"] * 100).round(1)

print("Activity by value quintile:")
churn_by_value

In [ ]:
# High-value churned customers (reactivation targets)
high_value_churned = df[(df["is_churned"] == True) & (df["value_quintile"] >= 4)].sort_values(
    "monetary_total", ascending=False
)[["company", "frequency", "monetary_total", "recency_days", "industry_sector", "region", "company_status"]]

print(f"High-value churned customers (Q4-Q5, >2yr inactive): {len(high_value_churned)}")
high_value_churned.head(20)

## 6. Industry × Region Opportunity Matrix

In [ ]:
# Cross-tab: top sectors × top regions by revenue
top_sectors = df.groupby("industry_sector")["monetary_total"].sum().nlargest(8).index.tolist()
top_regions = df.groupby("region")["monetary_total"].sum().nlargest(10).index.tolist()

filtered = df[df["industry_sector"].isin(top_sectors) & df["region"].isin(top_regions)]
matrix = filtered.pivot_table(
    index="industry_sector", columns="region",
    values="monetary_total", aggfunc="sum", fill_value=0
).round(0)

print("Revenue heatmap — Top sectors × Top regions (£):")
matrix

In [ ]:
# Company count version
count_matrix = filtered.pivot_table(
    index="industry_sector", columns="region",
    values="company", aggfunc="count", fill_value=0
)

print("Company count — Top sectors × Top regions:")
count_matrix

## 7. SIC Code Deep Dive

In [ ]:
# Explode SIC codes (companies can have multiple)
sic = df[df["sic_codes"].notna()].copy()
sic["sic_list"] = sic["sic_codes"].str.split(",")
sic_exploded = sic.explode("sic_list")
sic_exploded["sic_list"] = sic_exploded["sic_list"].str.strip()

# Top SIC codes by revenue
sic_revenue = sic_exploded.groupby("sic_list").agg(
    companies=("company", "nunique"),
    total_revenue=("monetary_total", "sum"),
    avg_revenue=("monetary_total", "mean"),
).sort_values("total_revenue", ascending=False).reset_index()

print(f"Unique SIC codes: {sic_exploded['sic_list'].nunique()}")
print("\nTop 25 SIC codes by revenue:")
sic_revenue.head(25)

## 8. Ideal Customer Profile (ICP)

In [ ]:
# Define "best" customers: top 20% by revenue, still active
revenue_80th = df["monetary_total"].quantile(0.80)
best = df[(df["monetary_total"] >= revenue_80th) & (df["is_active_12m"] == True)].copy()
rest = df[~df.index.isin(best.index)].copy()

print(f"Best customers (top 20% revenue + active): {len(best)}")
print(f"Rest: {len(rest)}")

# Compare profiles
compare_cols = [
    "frequency", "monetary_total", "monetary_mean", "avg_quantity",
    "avg_unit_price", "avg_margin", "avg_num_operations",
    "product_type_diversity", "orders_per_year", "tenure_days",
    "company_age_years",
]

comparison = pd.DataFrame({
    "best_mean": best[compare_cols].mean(),
    "best_median": best[compare_cols].median(),
    "rest_mean": rest[compare_cols].mean(),
    "rest_median": rest[compare_cols].median(),
}).round(2)
comparison["ratio"] = (comparison["best_mean"] / comparison["rest_mean"]).round(2)

print("\nBest customers vs rest:")
comparison

In [ ]:
# ICP industry breakdown
best_sectors = best["industry_sector"].value_counts()
rest_sectors = rest["industry_sector"].value_counts()

icp_sectors = pd.DataFrame({
    "best_count": best_sectors,
    "best_pct": (best_sectors / len(best) * 100).round(1),
    "rest_count": rest_sectors,
    "rest_pct": (rest_sectors / len(rest) * 100).round(1),
}).fillna(0)
icp_sectors["over_index"] = (icp_sectors["best_pct"] / icp_sectors["rest_pct"].replace(0, 0.1)).round(2)

print("ICP: Industry over-indexing (>1 = overrepresented in best customers):")
icp_sectors.sort_values("over_index", ascending=False)

In [ ]:
# ICP region breakdown
best_regions = best["region"].value_counts()
rest_regions = rest["region"].value_counts()

icp_regions = pd.DataFrame({
    "best_count": best_regions,
    "best_pct": (best_regions / len(best) * 100).round(1),
    "rest_count": rest_regions,
    "rest_pct": (rest_regions / len(rest) * 100).round(1),
}).fillna(0)
icp_regions["over_index"] = (icp_regions["best_pct"] / icp_regions["rest_pct"].replace(0, 0.1)).round(2)

print("ICP: Region over-indexing (top 15):")
icp_regions.sort_values("over_index", ascending=False).head(15)

In [ ]:
# ICP company size breakdown
best_size = best["accounts_type"].value_counts()
rest_size = rest["accounts_type"].value_counts()

icp_size = pd.DataFrame({
    "best_count": best_size,
    "best_pct": (best_size / best["accounts_type"].notna().sum() * 100).round(1),
    "rest_count": rest_size,
    "rest_pct": (rest_size / rest["accounts_type"].notna().sum() * 100).round(1),
}).fillna(0)
icp_size["over_index"] = (icp_size["best_pct"] / icp_size["rest_pct"].replace(0, 0.1)).round(2)

print("ICP: Company size over-indexing:")
icp_size.sort_values("over_index", ascending=False)

## 9. Company Governance & Filing Activity (Deep CH)

In [ ]:
# Deep CH feature coverage
deep_cols = ["officer_count", "active_officer_count", "director_count", "secretary_count",
             "filing_count", "recent_filing_days", "annual_accounts_count",
             "has_charges", "charge_count", "psc_count"]

print("Deep Companies House feature coverage:")
for c in deep_cols:
    if c in df.columns:
        n = df[c].notna().sum()
        print(f"  {c:30s} {n:>4}/{len(df)} ({100*n/len(df):.1f}%)")

print(f"\nKey stats:")
print(f"  Avg officers per company:     {df['officer_count'].mean():.1f}")
print(f"  Avg active directors:         {df['director_count'].mean():.1f}")
print(f"  Avg total filings:            {df['filing_count'].mean():.0f}")
print(f"  Companies with charges:       {df['has_charges'].sum()} ({df['has_charges'].mean():.1%})")
print(f"  Companies with PSC data:      {(df['psc_count'] > 0).sum()}")

In [ ]:
# Governance profile by value quintile — do bigger customers have more complex governance?
gov_by_value = df.groupby("value_quintile").agg(
    avg_officers=("officer_count", "mean"),
    avg_directors=("director_count", "mean"),
    avg_filings=("filing_count", "mean"),
    pct_with_charges=("has_charges", "mean"),
    avg_charges=("charge_count", "mean"),
    avg_psc=("psc_count", "mean"),
    avg_accounts_filed=("annual_accounts_count", "mean"),
).round(2)

gov_by_value["pct_with_charges"] = (gov_by_value["pct_with_charges"] * 100).round(1)

print("Governance profile by customer value quintile:")
gov_by_value

In [ ]:
# Companies with secured lending (charges) — do they spend more?
with_charges = df[df["has_charges"] == True]
without_charges = df[df["has_charges"] == False]

print("Companies with secured lending vs without:")
print(f"  {'':35s} {'With charges':>15s} {'Without':>15s}")
print(f"  {'Count':35s} {len(with_charges):>15,} {len(without_charges):>15,}")
print(f"  {'Avg revenue':35s} £{with_charges['monetary_total'].mean():>14,.0f} £{without_charges['monetary_total'].mean():>14,.0f}")
print(f"  {'Avg frequency':35s} {with_charges['frequency'].mean():>15.1f} {without_charges['frequency'].mean():>15.1f}")
print(f"  {'Avg company age (yrs)':35s} {with_charges['company_age_years'].mean():>15.1f} {without_charges['company_age_years'].mean():>15.1f}")
print(f"  {'Active (12m)':35s} {with_charges['is_active_12m'].mean():>14.1%} {without_charges['is_active_12m'].mean():>14.1%}")

In [ ]:
# Filing recency — are actively filing companies better customers?
filed = df[df["recent_filing_days"].notna()].copy()
filing_bins = [0, 90, 365, 730, float("inf")]
filing_labels = ["<3m", "3-12m", "1-2yr", "2yr+"]
filed["filing_recency"] = pd.cut(filed["recent_filing_days"], bins=filing_bins, labels=filing_labels)

filing_stats = filed.groupby("filing_recency", observed=True).agg(
    companies=("company", "count"),
    avg_revenue=("monetary_total", "mean"),
    avg_frequency=("frequency", "mean"),
    active_pct=("is_active_12m", "mean"),
).reset_index()
filing_stats["active_pct"] = (filing_stats["active_pct"] * 100).round(1)

print("Customer behaviour by filing recency:")
filing_stats

## 10. Web Presence Analysis

In [ ]:
# Web presence overview
print("Web presence coverage:")
print(f"  Has website:    {df['has_website'].sum()}/{len(df)} ({df['has_website'].mean():.1%})")
print(f"  Has HTTPS:      {df['has_https'].sum()}/{len(df)} ({df['has_https'].mean():.1%})")

print(f"\nDomain variants:")
print(df["domain_variant"].value_counts().to_string())

# Website vs no website
has_web = df[df["has_website"] == True]
no_web = df[df["has_website"] == False]

print(f"\nCompanies WITH website vs WITHOUT:")
print(f"  {'':35s} {'Has website':>15s} {'No website':>15s}")
print(f"  {'Count':35s} {len(has_web):>15,} {len(no_web):>15,}")
print(f"  {'Avg revenue':35s} £{has_web['monetary_total'].mean():>14,.0f} £{no_web['monetary_total'].mean():>14,.0f}")
print(f"  {'Avg frequency':35s} {has_web['frequency'].mean():>15.1f} {no_web['frequency'].mean():>15.1f}")
print(f"  {'Avg margin':35s} {has_web['avg_margin'].mean():>14.1f}% {no_web['avg_margin'].mean():>14.1f}%")
print(f"  {'Active (12m)':35s} {has_web['is_active_12m'].mean():>14.1%} {no_web['is_active_12m'].mean():>14.1%}")
print(f"  {'Avg company age':35s} {has_web['company_age_years'].mean():>14.1f} {no_web['company_age_years'].mean():>14.1f}")

In [ ]:
# HTTPS adoption by sector — proxy for digital maturity
web_by_sector = df[df["industry_sector"].notna()].groupby("industry_sector").agg(
    companies=("company", "count"),
    has_website_pct=("has_website", "mean"),
    has_https_pct=("has_https", "mean"),
    avg_revenue=("monetary_total", "mean"),
).sort_values("has_https_pct", ascending=False)

web_by_sector["has_website_pct"] = (web_by_sector["has_website_pct"] * 100).round(1)
web_by_sector["has_https_pct"] = (web_by_sector["has_https_pct"] * 100).round(1)

print("Web presence by industry sector:")
web_by_sector

## 11. Growth & Retention Signals

In [ ]:
# Revenue trend by industry
trend_by_sector = df[df["industry_sector"].notna()].groupby("industry_sector").agg(
    companies=("company", "count"),
    avg_trend_pct=("revenue_trend_pct", "mean"),
    avg_recent_share=("recent_share_pct", "mean"),
    active_pct=("is_active_12m", "mean"),
    churned_pct=("is_churned", "mean"),
).round(2).sort_values("avg_recent_share", ascending=False)

trend_by_sector["active_pct"] = (trend_by_sector["active_pct"] * 100).round(1)
trend_by_sector["churned_pct"] = (trend_by_sector["churned_pct"] * 100).round(1)

print("Growth & retention by sector:")
trend_by_sector

In [ ]:
# Top growing customers (positive revenue trend + still active)
growing = df[(df["revenue_trend_pct"] > 0) & (df["is_active_12m"] == True) & (df["frequency"] >= 3)]
growing_top = growing.nlargest(20, "revenue_trend_pct")[
    ["company", "frequency", "monetary_total", "revenue_trend_pct",
     "recent_share_pct", "industry_sector", "region"]
]

print(f"Growing active customers (≥3 orders, positive trend): {len(growing)}")
print("\nTop 20 by growth rate:")
growing_top

## 12. Feature Correlations for ML

In [ ]:
# Correlation of external features with revenue
numeric_cols = df.select_dtypes(include=["float64", "int64", "bool"]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ["value_quintile"]]

correlations = df[numeric_cols].corr()["monetary_total"].drop("monetary_total").sort_values(ascending=False)

print("Top features correlated with total revenue:")
print("\nPositive:")
for feat, corr in correlations.head(15).items():
    bar = "█" * int(abs(corr) * 30)
    print(f"  {feat:45s} {corr:+.3f}  {bar}")

print("\nNegative:")
for feat, corr in correlations.tail(10).items():
    bar = "░" * int(abs(corr) * 30)
    print(f"  {feat:45s} {corr:+.3f}  {bar}")

In [ ]:
# Summary
deep_cols = ["officer_count", "active_officer_count", "director_count", "secretary_count",
             "filing_count", "recent_filing_days", "annual_accounts_count",
             "has_charges", "charge_count", "psc_count"]
web_cols = ["has_website", "website_domain", "has_https", "http_status", "domain_variant"]
ch_cols = ["ch_company_name", "company_number", "company_type", "company_status",
           "sic_codes", "industry_sector", "incorporation_date", "company_age_years",
           "region", "accounts_type"]

print("="*60)
print("ENRICHED DATASET SUMMARY")
print("="*60)
print(f"Companies:             {len(df)}")
print(f"Total features:        {len(df.columns)}")
print(f"  Internal (orders):   67")
print(f"  CH basic:            {len(ch_cols)}")
print(f"  CH deep:             {len(deep_cols)}")
print(f"  Web presence:        {len(web_cols)}")
print(f"  Derived:             {len(df.columns) - 67 - len(ch_cols) - len(deep_cols) - len(web_cols)}")
print(f"")
print(f"CH match rate:         {df['company_number'].notna().mean():.1%}")
print(f"SIC coverage:          {df['sic_codes'].notna().mean():.1%}")
print(f"Region coverage:       {df['region'].notna().mean():.1%}")
print(f"Website coverage:      {df['has_website'].mean():.1%}")
print(f"Deep CH coverage:      {df['officer_count'].notna().mean():.1%}")
print(f"")
print(f"Active (12m):          {df['is_active_12m'].sum()} ({df['is_active_12m'].mean():.1%})")
print(f"Churned (>2yr):        {df['is_churned'].sum()} ({df['is_churned'].mean():.1%})")
print(f"Top sector:            {df['industry_sector'].value_counts().index[0]}")
print(f"Top region:            {df['region'].value_counts().index[0]}")
print(f"\nReady for clustering in: notebooks/customer_segmentation.ipynb")